# 0) IMPORTS

In [1]:
# Setup autoreload - automatically reload modules when they change
%load_ext autoreload
%autoreload 2

In [2]:
# Verify environment setup
import os
import sys

print(f"Python version: {sys.version}")
print(f"Python executable: {sys.executable}")
print(f"\nEnvironment variables loaded:")
print(f"OPENAI_API_KEY: {'✓ Set' if os.getenv('OPENAI_API_KEY') else '✗ Not set'}")
print(f"MASSIVE.COM - ex.POLYGON_API_KEY: {'✓ Set' if os.getenv('POLYGON_API_KEY') else '✗ Not set'}")
print(f"XAI_API_KEY: {'✓ Set' if os.getenv('XAI_API_KEY') else '✗ Not set'}")
print(f"SEC_IDENTITY_EMAIL: {'✓ Set' if os.getenv('SEC_IDENTITY_EMAIL') else '✗ Not set'}")


Python version: 3.12.7 (main, Oct  1 2024, 02:05:46) [Clang 16.0.0 (clang-1600.0.26.3)]
Python executable: /Users/realmistic/Documents/stocks-scoring-agent/.venv/bin/python

Environment variables loaded:
OPENAI_API_KEY: ✓ Set
MASSIVE.COM - ex.POLYGON_API_KEY: ✓ Set
XAI_API_KEY: ✓ Set
SEC_IDENTITY_EMAIL: ✓ Set


# 1) TEST: Checking news retrieval for the recent 5k records 
* The datasource is Massive.com (previously known as Polygon.io)
* 2 tools defined: 
  * search_news_by_ticker (not just YahooFinance source)
  * search_news_by_query (it can find a wider set of news to expand the topic)


In [3]:
from datetime import datetime, timezone
now = datetime.now(timezone.utc).strftime("%Y-%m-%d")
print (f'Time now = {now}')

# The exact time of running the Colab (in UTC, timezone-aware)
now_right_format = datetime.now(timezone.utc).isoformat(timespec='milliseconds')
print(f'Time now in the right format (for API) = {now_right_format}')

Time now = 2026-06-14
Time now in the right format (for API) = 2026-06-14T13:06:27.443+00:00


In [4]:
import requests
import pandas as pd
import os

# MASSIVE.COM is previously known as Polygon.io
# https://massive.com/docs/rest/stocks/news
# https://massive.com/blog/api-pagination-patterns
# API CALL : # https://api.massive.com/v2/reference/news?order=desc&limit=1000&sort=published_utc&apiKey=<your key> or POLYGON_API_KEY
      # need to get 200 OK status

# check the API KEY is imported correctly from the .envrc file
print(len(f'Length of the POLYGON_API_KEY = {os.getenv("POLYGON_API_KEY")}'))

POLYGON_API_KEY = os.getenv('POLYGON_API_KEY')

# retrieve max 1000 news via one API call
def get_one_chunk_of_news_polygon_io(api_key = POLYGON_API_KEY, news_limit=1000, max_date = now):
#OLD:   url = f"https://api.polygon.io/v2/reference/news?order=desc&limit={news_limit}&sort=published_utc&published_utc.lt={max_date}&apiKey={api_key}"
  url = f"https://api.massive.com/v2/reference/news?order=desc&limit={news_limit}&sort=published_utc&published_utc.lt={max_date}&apiKey={api_key}"


  # https://www.nylas.com/blog/use-python-requests-module-rest-apis/ - Python for rest APIs
  # try/catch for HTTP requests: https://stackoverflow.com/questions/16511337/correct-way-to-try-except-using-python-requests-module
  try:
      url_sanitized = url.split("&apiKey=")[0] + "&apiKey=<your key>"
      print(f'trying the API call : {url_sanitized}')
      r = requests.get(url, timeout=3)
      r.raise_for_status()
  except requests.exceptions.HTTPError as errh:
      print ("Http Error:",errh)
  except requests.exceptions.ConnectionError as errc:
      print ("Error Connecting:",errc)
  except requests.exceptions.Timeout as errt:
      print ("Timeout Error:",errt)
  except requests.exceptions.RequestException as err:
      print ("OOps: Something Else",err)

  data = r.json()


  # Check if 'results' key exists in the response
  if 'results' in data:
      # If it exists, proceed with normalization
      df_nested_list = pd.json_normalize(data, record_path=['results'])
  else:
      # If not, print the response keys for debugging and create an empty DataFrame
      print(f"The 'results' key was not found in the response. Available keys are: {data.keys()}")
      df_nested_list = pd.DataFrame() # or handle it differently based on the new structure

  return df_nested_list

def get_all_news(api_calls_left = 5, api_key = POLYGON_API_KEY, news_limit=1000, max_date = now):
  all_news = None
  for i in range(api_calls_left):
    cur = get_one_chunk_of_news_polygon_io(api_key = api_key, news_limit = news_limit, max_date = max_date)
    if all_news is None:
      all_news = cur
    else:
      all_news = pd.concat([all_news,cur], ignore_index=True, axis=0) #stacking dataframes

    max_date = cur.published_utc.min() #update max_date of the news
  return all_news

64


In [5]:
# test getting all news
# 5 calls per minute limit for a free account - all recent news (5000)
all_news = get_all_news()

trying the API call : https://api.massive.com/v2/reference/news?order=desc&limit=1000&sort=published_utc&published_utc.lt=2026-06-14&apiKey=<your key>
trying the API call : https://api.massive.com/v2/reference/news?order=desc&limit=1000&sort=published_utc&published_utc.lt=2026-06-09T17:28:49Z&apiKey=<your key>
trying the API call : https://api.massive.com/v2/reference/news?order=desc&limit=1000&sort=published_utc&published_utc.lt=2026-06-04T10:00:00Z&apiKey=<your key>
trying the API call : https://api.massive.com/v2/reference/news?order=desc&limit=1000&sort=published_utc&published_utc.lt=2026-05-29T22:15:00Z&apiKey=<your key>
trying the API call : https://api.massive.com/v2/reference/news?order=desc&limit=1000&sort=published_utc&published_utc.lt=2026-05-25T18:56:00Z&apiKey=<your key>


In [6]:
all_news.head()

,id,title,author,published_utc,article_url,tickers,image_url,description,keywords,insights,publisher.name,publisher.homepage_url,publisher.logo_url,publisher.favicon_url,amp_url
0,b2f3a113f375b61f1bc288a1d08cc27adc88ef79b8482b...,"ROSEN, NATIONAL INVESTOR COUNSEL, Encourages V...",Rosen Law Firm,2026-06-13T23:50:00Z,https://www.globenewswire.com/news-release/202...,[VERI],https://ml.globenewswire.com/Resource/Download...,Rosen Law Firm is urging investors who purchas...,"[securities class action, financial restatemen...","[{'ticker': 'VERI', 'sentiment': 'negative', '...",GlobeNewswire Inc.,https://www.globenewswire.com,https://s3.massive.com/public/assets/news/logo...,https://s3.massive.com/public/assets/news/favi...,NaN
1,8660ae9a9565114ac6b1135157c21f6d1790a3eca72b10...,"SpaceX IPO Sets the Stage, These Five Tech Com...",Crispus Nyaga,2026-06-13T23:26:37Z,https://www.benzinga.com/markets/tech/26/06/53...,"[SPCX, NVDA, ORCL, ORCLpD]",https://cdn.benzinga.com/cdn-cgi/image/width=1...,"Following SpaceX's successful $75 billion IPO,...","[IPO, SpaceX, Anthropic, OpenAI, tech companie...","[{'ticker': 'SPCX', 'sentiment': 'positive', '...",Benzinga,https://www.benzinga.com/,https://s3.massive.com/public/assets/news/logo...,https://s3.massive.com/public/assets/news/favi...,NaN
2,4d52d542b8a2d8d73b5a0010ef06793b64124a98dc0def...,How Insurance Companies Turn Their Premiums In...,Reuben Gregg Brewer,2026-06-13T23:15:00Z,https://www.fool.com/investing/2026/06/13/how-...,"[BRK.A, BRK.B, PGR, MKL, BN, BNH, BNJ]",https://g.foolcdn.com/image/?url=https%3A%2F%2...,Insurance companies generate billions in profi...,"[insurance float, investment income, premium i...","[{'ticker': 'BRK.A', 'sentiment': 'positive', ...",The Motley Fool,https://www.fool.com/,https://s3.massive.com/public/assets/news/logo...,https://s3.massive.com/public/assets/news/favi...,NaN
3,12b48538e163af0c55da8843aa5dab211becdac2941733...,How Safe is Bristol Myers Squibb's Dividend? H...,Reuben Gregg Brewer,2026-06-13T22:15:00Z,https://www.fool.com/investing/2026/06/13/how-...,"[BMY, CELGr, PFE]",https://g.foolcdn.com/image/?url=https%3A%2F%2...,Bristol Myers Squibb offers an attractive 4.5%...,"[dividend safety, pharmaceutical sector, paten...","[{'ticker': 'BMY', 'sentiment': 'positive', 's...",The Motley Fool,https://www.fool.com/,https://s3.massive.com/public/assets/news/logo...,https://s3.massive.com/public/assets/news/favi...,NaN
4,9cc0eb5cf81786ffdfc025d8b10096d2087bc931635ce0...,Is Palantir Stock Ripe for a Rebound?,Adria Cimino,2026-06-13T22:10:00Z,https://www.fool.com/investing/2026/06/13/is-p...,"[PLTR, NVDA, GOOG, GOOGL, GOOGM, GOOGN]",https://g.foolcdn.com/image/?url=https%3A%2F%2...,Palantir Technologies has experienced strong r...,"[artificial intelligence, AI platform, valuati...","[{'ticker': 'PLTR', 'sentiment': 'neutral', 's...",The Motley Fool,https://www.fool.com/,https://s3.massive.com/public/assets/news/logo...,https://s3.massive.com/public/assets/news/favi...,NaN


In [7]:
all_news_json = all_news.to_dict(orient='records')

In [8]:
len(all_news_json)

5000

In [9]:
# Key features:

#   1. No chunking needed - Each news article is already a discrete document
#   2. Text search - Searches across title, description, keywords, author
#   3. Exact ticker matching - Function to find articles with specific ticker in tickers field
#   4. Multi-ticker support - Search for multiple tickers at once
#   5. Date filtering - Can filter news by publication date
#   6. Boosting - Title matches get higher priority than description

#   Note on chunking:
#   Unlike the podcast data (long markdown documents), news articles are already atomic units. Each article is a complete piece of information, so no chunking is
#   needed. The tickers field already provides precise ticker associations.

In [10]:

# Import minsearch
from minsearch import Index

# Step 1: Convert DataFrame to list of dicts (if not already done)
news_documents = all_news.to_dict(orient='records')

print(f"Total news articles: {len(news_documents)}")
print(f"Fields per article: {len(news_documents[0])}")
print(f"Sample article keys: {list(news_documents[0].keys())}")

Total news articles: 5000
Fields per article: 15
Sample article keys: ['id', 'title', 'author', 'published_utc', 'article_url', 'tickers', 'image_url', 'description', 'keywords', 'insights', 'publisher.name', 'publisher.homepage_url', 'publisher.logo_url', 'publisher.favicon_url', 'amp_url']


In [11]:
from tqdm import tqdm
  # Step 2: Preprocess documents - convert list fields to strings
print("\nPreprocessing documents...")
for doc in tqdm(news_documents, desc="Converting fields"):
    # Convert list fields to comma-separated strings
    if isinstance(doc.get('tickers'), list):
        doc['tickers'] = ', '.join(doc['tickers'])

    if isinstance(doc.get('keywords'), list):
        doc['keywords'] = ', '.join(doc['keywords'])

    # Ensure text fields are strings
    for field in ['title', 'description', 'author']:
        if doc.get(field) is None:
            doc[field] = ''
        elif not isinstance(doc.get(field), str):
            doc[field] = str(doc[field])

print("✓ Preprocessing complete")


Preprocessing documents...


Converting fields: 100%|██████████| 5000/5000 [00:00<00:00, 777356.36it/s]

✓ Preprocessing complete


In [12]:

  # Step 3: Create Index with relevant text fields
news_index = Index(
    text_fields=["title", "description", "keywords", "author", "tickers"],
    keyword_fields=["published_utc", "publisher.name"]
)

# Step 4: Fit the index with preprocessed documents
print("\nBuilding search index...")
news_index.fit(news_documents)
print(f"✓ Index created with {len(news_documents)} articles")


Building search index...
✓ Index created with 5000 articles


In [13]:
# Step 5: Search function to retrieve news for a ticker
def search_news_by_ticker(ticker, num_results=30):
    """
      Search for news articles related to a specific ticker.
      
      Args:
          ticker: Stock ticker symbol (e.g., 'TSLA', 'AAPL')
          num_results: Number of results to return
      
      Returns:
          List of news articles matching the ticker
    """
    results = news_index.search(
        query=ticker,
        num_results=num_results,
        boost_dict={
              'tickers': 5.0,      # Highest boost for ticker field
              'title': 3.0,         # High boost for title
              'description': 1.5,   # Medium boost for description
              'keywords': 1.0       # Standard boost for keywords
          }
      )
    return results

In [14]:
# Step 6: Search function to retrieve news for a search query
def search_news_by_query(query, num_results=30):
    """
      Search for news articles related to a specific query.
      
      Args:
          query: Search query (e.g., 'Tesla', 'AI robotics')
          num_results: Number of results to return
      
      Returns:
          List of news articles matching the ticker
    """
    results = news_index.search(
        query=query,
        num_results=num_results,
        boost_dict={
              'tickers': 1.0,      # boost for ticker field
              'title': 3.0,         # High boost for title
              'description': 5,   # Highest boost for description
              'keywords': 5       # Highest boost for keywords
          }
      )
    return results

In [15]:

# Step 7: Test - Get news for TSLA
print("\n" + "="*70)
print("Testing search for TSLA...")
print("="*70)

tsla_news = search_news_by_ticker("TSLA", num_results=5)

print(f"\nFound {len(tsla_news)} news articles for TSLA:\n")
for i, article in enumerate(tsla_news, 1):
    print(f"{i}. {article['title']}")
    print(f"   Published: {article['published_utc']}")
    print(f"   Tickers: {article['tickers']}")
    print(f"   Description: {article['description'][:100]}...")
    print()


Testing search for TSLA...

Found 5 news articles for TSLA:

1. Crypto Markets Are Already Pricing SPCX: What Happens At IPO?
   Published: 2026-06-04T16:09:45Z
   Tickers: TSLA
   Description: SpaceX is preparing for a major IPO in June 2026 targeting a $1.8 trillion valuation and $70-75 bill...

2. Will SpaceX Merge With Tesla? The Answer Might Be Hiding in Plain Sight.
   Published: 2026-06-08T17:17:00Z
   Tickers: TSLA
   Description: SpaceX's updated S-1 IPO filing contains revised language suggesting the company may issue significa...

3. Tesla Investors Finally Get A Benchmark For The Musk Premium
   Published: 2026-06-01T17:19:11Z
   Tickers: TSLA
   Description: SpaceX's anticipated IPO will provide investors with a second publicly traded Musk-led company, allo...

4. Here Are 7 Important Things Investors Learned from SpaceX's S-1 Filing
   Published: 2026-05-22T21:20:00Z
   Tickers: TSLA
   Description: SpaceX filed its S-1 prospectus ahead of its IPO, seeking a $2 trillion 

In [16]:
# Step 8: search news by query
print("\n" + "="*70)
print("Testing search by query...")
print("="*70)

query = "Tesla competitors analysis EV margins autonomy AI robotics China market share outlook"
tsla_competitors_news = search_news_by_query(query, num_results=25)

print(f"\nFound {len(tsla_competitors_news)} news articles for TSLA competitors:\n")
for i, article in enumerate(tsla_competitors_news, 1):
    print(f"{i}. {article['title']}")
    print(f"   Published: {article['published_utc']}")
    print(f"   Tickers: {article['tickers']}")
    print(f"   Description: {article['description'][:100]}...")
    print()


Testing search by query...

Found 25 news articles for TSLA competitors:

1. EV Sales Skyrocket: Tesla Stock Surges On 67% European Boom, Blockbuster SpaceX IPO Rumors
   Published: 2026-05-27T15:40:06Z
   Tickers: TSLA, BYDDY, GELHY
   Description: Tesla stock gained 1.11% on Wednesday, driven by strong European EV sales growth (67.2% year-over-ye...

2. Following Trump-Xi Talks, Elon Musk's Tesla Launches Full Self-Driving In China After Years Of Uncertainty
   Published: 2026-05-21T10:26:10Z
   Tickers: TSLA, BYDDY, BA, BApA
   Description: Tesla has launched its Full Self-Driving (FSD) service in China, expanding to 10 countries total, fo...

3. Tesla’s European Recovery Is Real, But BYD Defines the Limit
   Published: 2026-05-27T07:07:00Z
   Tickers: TSLA, BYDDY
   Description: Tesla shows genuine recovery in European EV registrations with three consecutive months of year-on-y...

4. Is China's EV Boom Losing Steam? Tesla Rival Nio's CEO Sounds The Alarm
   Published: 2026-05-28T

# 2) TEST: Checking Yahoo Finance API endpoints

In [17]:
# https://github.com/ranaroussi/yfinance
# https://algotrading101.com/learn/yfinance-guide/

import yfinance as yf
ticker="TSLA"
ticker_obj = yf.Ticker(ticker)

In [18]:
# daily stats for a stock: Open, High, Low, Close, Volume
historical_prices = ticker_obj.history(period="2y", interval="1d")

# show only date part in the index
historical_prices.index = historical_prices.index.date

historical_prices.head()

,Open,High,Low,Close,Volume,Dividends,Stock Splits
2024-06-13,188.389999,191.080002,181.229996,182.470001,118984100,0.0,0.0
2024-06-14,185.800003,186.000000,176.919998,178.009995,82038200,0.0,0.0
2024-06-17,177.919998,188.809998,177.000000,187.440002,109786100,0.0,0.0
2024-06-18,186.559998,187.199997,182.369995,184.860001,68982300,0.0,0.0
2024-06-20,184.679993,185.210007,179.660004,181.570007,55893100,0.0,0.0


In [19]:
# Analyst price targets
ticker_obj.get_analyst_price_targets()

{'current': 406.43,
 'high': 600.0,
 'low': 123.0,
 'mean': 420.54633,
 'median': 460.0}

In [20]:
# Dividends or Stock splits history
ticker_obj.actions

,Dividends,Stock Splits
Date,,
2020-08-31 00:00:00-04:00,0.0,5.0
2022-08-25 00:00:00-04:00,0.0,3.0


In [21]:
# Next earnings date + targets on EPS (Earnings)/revenue
ticker_obj.calendar

{'Earnings Date': [datetime.date(2026, 7, 22)],
 'Earnings High': 0.59,
 'Earnings Low': 0.26483,
 'Earnings Average': 0.45414,
 'Revenue High': 26059000000,
 'Revenue Low': 22800000000,
 'Revenue Average': 24634836150}

In [22]:
# FIN STATEMENTS -- too much of info, not needed for now
# ticker_obj.get_balancesheet()
# ticker_obj.get_cash_flow()
# ticker_obj.get_income_stmt()
# same as income stmt: ticker_obj.get_financials()

In [23]:
# EPS estimates vs. numberOfAnalysts
ticker_obj.get_earnings_estimate()

,avg,low,high,yearAgoEps,numberOfAnalysts,growth
period,,,,,,
0q,0.45414,0.26483,0.59,0.40000,25,0.1353
+1q,0.53750,0.29981,0.83,0.50000,25,0.0750
0y,2.05937,1.35491,3.19,1.66000,34,0.2406
+1y,2.49995,1.46400,3.65,2.05937,30,0.2139


In [24]:
# How analysts have revised their EPS estimates over time?
ticker_obj.get_eps_revisions()

,upLast7days,upLast30days,downLast30days,downLast7Days
period,,,,
0q,1,2,1,0
+1q,1,1,1,0
0y,1,1,2,0
+1y,0,0,3,1


In [25]:
# This section retrieves consensus analyst growth expectations from Yahoo Finance for the selected ticker. 
# The values represent estimated earnings growth and are expressed in decimal form, 
# where positive values indicate expected growth and negative values indicate contraction. 
# Estimates are provided across multiple time horizons: 0q refers to the current fiscal quarter, 
# +1q to the next fiscal quarter, 0y to the current fiscal year, and +1y to the next fiscal year.
#  Short-term quarterly estimates tend to be more volatile, while annual estimates provide a broader view of expected performance.

ticker_obj.get_growth_estimates()

,stockTrend,indexTrend
period,,
0q,0.1353,0.2598
+1q,0.0750,0.2028
0y,0.2406,0.2308
+1y,0.2139,0.1704
LTG,NaN,0.1220


In [26]:
ticker_obj.get_earnings_history()

,epsActual,epsEstimate,epsDifference,surprisePercent
quarter,,,,
2025-06-30,0.40,0.40391,-0.004,-0.0097
2025-09-30,0.50,0.55885,-0.060,-0.1053
2025-12-31,0.50,0.45061,0.050,0.1096
2026-03-31,0.41,0.34999,0.060,0.1715


In [27]:
import pandas as pd
news = pd.DataFrame(ticker_obj.get_news())
news.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   id       10 non-null     object
 1   content  10 non-null     object
dtypes: object(2)
memory usage: 292.0+ bytes


In [28]:
news

,id,content
0,8ef49b96-5099-399d-b765-1e95749d4f6f,"{'id': '8ef49b96-5099-399d-b765-1e95749d4f6f',..."
1,eafe3e98-b47c-3e2a-9345-a01348e43c03,"{'id': 'eafe3e98-b47c-3e2a-9345-a01348e43c03',..."
2,3cdded0e-83d4-37e2-bafa-229d9c2a642f,"{'id': '3cdded0e-83d4-37e2-bafa-229d9c2a642f',..."
3,846e88d0-53ac-3b91-8527-c9c805463568,"{'id': '846e88d0-53ac-3b91-8527-c9c805463568',..."
4,0679fbd1-f17d-3a74-8d34-f094d76bae10,"{'id': '0679fbd1-f17d-3a74-8d34-f094d76bae10',..."
5,6c53b91b-d39c-3259-8e3d-184a3c3652a8,"{'id': '6c53b91b-d39c-3259-8e3d-184a3c3652a8',..."
6,d06aeb26-91e1-3940-8138-aa75d4ef3209,"{'id': 'd06aeb26-91e1-3940-8138-aa75d4ef3209',..."
7,2729cce8-9d0e-39e4-8664-4ebbc474140b,"{'id': '2729cce8-9d0e-39e4-8664-4ebbc474140b',..."
8,e54dbe29-f9ed-3af4-846a-3e1c4d735f61,"{'id': 'e54dbe29-f9ed-3af4-846a-3e1c4d735f61',..."
9,a687efe7-066a-3e64-91b2-1124f52354d0,"{'id': 'a687efe7-066a-3e64-91b2-1124f52354d0',..."


In [29]:
import pandas as pd

# Create a new DataFrame from the 'content' column of the 'news' DataFrame
news_df = pd.DataFrame(news['content'].tolist())

# Display the first few rows of the new DataFrame
display(news_df.head())

,id,contentType,title,description,summary,pubDate,displayTime,isHosted,bypassModal,previewUrl,thumbnail,provider,canonicalUrl,clickThroughUrl,metadata,finance,storyline
0,8ef49b96-5099-399d-b765-1e95749d4f6f,STORY,OpenAI Just Launched a Robotics Division. Shou...,,ChatGPT's parent isn't an immediate threat. Th...,2026-06-14T12:05:00Z,2026-06-14T12:05:00Z,True,False,None,{'originalUrl': 'https://media.zenfs.com/en/mo...,"{'displayName': 'Motley Fool', 'url': 'http://...",{'url': 'https://www.fool.com/investing/2026/0...,{'url': 'https://finance.yahoo.com/sectors/tec...,{'editorsPick': False},"{'premiumFinance': {'isPremiumNews': False, 'i...",None
1,eafe3e98-b47c-3e2a-9345-a01348e43c03,STORY,3 Genius Stocks I'm Buying Instead of the Spac...,,"SpaceX IPO may be popular, but there are bette...",2026-06-14T08:05:00Z,2026-06-14T08:05:00Z,True,False,None,{'originalUrl': 'https://media.zenfs.com/en/mo...,"{'displayName': 'Motley Fool', 'url': 'http://...",{'url': 'https://www.fool.com/investing/2026/0...,{'url': 'https://finance.yahoo.com/markets/sto...,{'editorsPick': False},"{'premiumFinance': {'isPremiumNews': False, 'i...",None
2,3cdded0e-83d4-37e2-bafa-229d9c2a642f,STORY,SpaceX Surge Further Boosts Saudi Billionaire ...,,(Bloomberg) -- Friday’s debut share surge for ...,2026-06-14T07:33:34Z,2026-06-14T07:33:34Z,True,False,None,None,"{'displayName': 'Bloomberg', 'url': 'https://w...",{'url': 'https://finance.yahoo.com/markets/sto...,{'url': 'https://finance.yahoo.com/markets/sto...,{'editorsPick': False},"{'premiumFinance': {'isPremiumNews': False, 'i...",None
3,846e88d0-53ac-3b91-8527-c9c805463568,STORY,Can Rivian Beat Tesla in the Long Term?,,Rivian began delivering its affordable R2 flee...,2026-06-14T02:53:00Z,2026-06-14T02:53:00Z,True,False,None,{'originalUrl': 'https://media.zenfs.com/en/mo...,"{'displayName': 'Motley Fool', 'url': 'http://...",{'url': 'https://www.fool.com/investing/2026/0...,{'url': 'https://finance.yahoo.com/markets/sto...,{'editorsPick': False},"{'premiumFinance': {'isPremiumNews': False, 'i...",None
4,0679fbd1-f17d-3a74-8d34-f094d76bae10,STORY,Elon Musk Is Now the World's First Trillionair...,,A second Musk-led megacap is now publicly trad...,2026-06-14T01:31:00Z,2026-06-14T01:31:00Z,True,False,None,{'originalUrl': 'https://media.zenfs.com/en/mo...,"{'displayName': 'Motley Fool', 'url': 'http://...",{'url': 'https://www.fool.com/investing/2026/0...,{'url': 'https://finance.yahoo.com/markets/sto...,{'editorsPick': False},"{'premiumFinance': {'isPremiumNews': False, 'i...",None


In [30]:
# summary of news articles for the ticker (Yahoo Finance)
from pprint import pprint 
for e in news_df.summary:
    pprint(e)
    print('-----')

("ChatGPT's parent isn't an immediate threat. The fact that it's planning on "
 "entering the market at all, however, underscores the idea that Tesla isn't "
 'going to be in this business all by itself.')
-----
'SpaceX IPO may be popular, but there are better options available.'
-----
('(Bloomberg) -- Friday’s debut share surge for SpaceX is bolstering the '
 'fortunes of one of Saudi Arabia’s richest men. Most Read from BloombergWhy '
 'Musk Raced to Take SpaceX Public in the World’s Biggest IPOTrump Eyes Sunday '
 'Iran Deal But Tehran Says Still Reviewing TextAnthropic Shuts Down Mythos '
 'Access After Sweeping US OrderUS Wants to Avoid Congressional Vote on Trade '
 'Deal, Carney SaysAir India Plans to Downsize With Owner Tata Balking at '
 'LossesPrince Alwaleed bin Talal’s investment firm Kingd')
-----
('Rivian began delivering its affordable R2 fleet this week in a push to bring '
 'the company to the mainstream.')
-----
('A second Musk-led megacap is now publicly traded -- an

In [31]:
# Fast info about the stock
ticker_obj.get_fast_info().items()

[('currency', 'USD'),
 ('dayHigh', 406.67999267578125),
 ('dayLow', 386.760009765625),
 ('exchange', 'NMS'),
 ('fiftyDayAverage', 398.3000012207031),
 ('lastPrice', 406.42999267578125),
 ('lastVolume', 63514500),
 ('marketCap', 1526438825382.7869),
 ('open', 399.489990234375),
 ('previousClose', 398.9999),
 ('quoteType', 'EQUITY'),
 ('regularMarketPreviousClose', 399.1499938964844),
 ('shares', 3755723871),
 ('tenDayAverageVolume', 49458150),
 ('threeMonthAverageVolume', 58958131),
 ('timezone', 'America/New_York'),
 ('twoHundredDayAverage', 415.6944003295898),
 ('yearChange', 0.23486156426391933),
 ('yearHigh', 498.8299865722656),
 ('yearLow', 288.7699890136719)]

In [32]:
import json

In [33]:
# a lot of info - potentially useful fields:
# website, industry, sector, fullTimeEmployees, companyOfficers, previousClose, open, dayLow, dayHigh, 
# beta, forwardPE, trailingPE, volume, averageVolume, averageVolume10days, marketCap, fiftyTwoWeekLow, 
# fiftyTwoWeekHigh, allTimeHigh, allTimeLow, priceToSalesTrailing12Months, fiftyDayAverage, twoHundredDayAverage
# profitMargins, sharesPercentSharesOut, heldPercentInsiders, heldPercentInstitutions, shortRatio, bookValue, priceToBook
# earningsQuarterlyGrowth, trailingEps, forwardEps, 52WeekChange, currentPrice,
#   targetHighPrice, targetLowPrice, targetMeanPrice, targetMedianPrice, recommendationMean,
# recommendationKey, numberOfAnalystOpinions, totalCashPerShare,totalCashPerShare, returnOnAssets, returnOnEquity,
# earningsGrowth, revenueGrowth, grossMargins, ebitdaMargins, operatingMargins, region, fullExchangeName, regularMarketDayRange, fiftyTwoWeekRange,
# fiftyTwoWeekHighChange, cryptoTradeable, displayName, trailingPegRatio
info = ticker_obj.get_info()
print(json.dumps(info, indent=2, default=str))

{
  "address1": "1 Tesla Road",
  "city": "Austin",
  "state": "TX",
  "zip": "78725",
  "country": "United States",
  "phone": "512 516 8177",
  "website": "https://www.tesla.com",
  "industry": "Auto Manufacturers",
  "industryKey": "auto-manufacturers",
  "industryDisp": "Auto Manufacturers",
  "sector": "Consumer Cyclical",
  "sectorKey": "consumer-cyclical",
  "sectorDisp": "Consumer Cyclical",
  "longBusinessSummary": "Tesla, Inc. designs, develops, manufactures, leases, and sells electric vehicles, and energy generation and storage systems in the United States, China, and internationally. The company operates in two segments, Automotive; and Energy Generation and Storage. The company offers electric vehicles, as well as sells automotive regulatory credits; and non-warranty maintenance services and collision, automotive insurance services, as well as part sales and retail merchandise sale. It also provides sedans and sport utility vehicles through direct and used vehicle sales, a

In [34]:
# more history metadata - not sure we need this now
import json

metadata = ticker_obj.get_history_metadata()
print(json.dumps(metadata, indent=2, default=str))

{
  "currency": "USD",
  "symbol": "TSLA",
  "exchangeName": "NMS",
  "fullExchangeName": "NasdaqGS",
  "instrumentType": "EQUITY",
  "firstTradeDate": 1277818200,
  "regularMarketTime": 1781294400,
  "hasPrePostMarketData": true,
  "gmtoffset": -14400,
  "timezone": "EDT",
  "exchangeTimezoneName": "America/New_York",
  "regularMarketPrice": 406.43,
  "fiftyTwoWeekHigh": 498.83,
  "fiftyTwoWeekLow": 288.77,
  "regularMarketDayHigh": 406.68,
  "regularMarketDayLow": 386.76,
  "regularMarketVolume": 63652286,
  "longName": "Tesla, Inc.",
  "shortName": "Tesla, Inc.",
  "chartPreviousClose": 391.0,
  "previousClose": 399.15,
  "scale": 3,
  "priceHint": 2,
  "currentTradingPeriod": {
    "pre": {
      "timezone": "EDT",
      "start": 1781251200,
      "end": 1781271000,
      "gmtoffset": -14400
    },
    "regular": {
      "timezone": "EDT",
      "start": 1781271000,
      "end": 1781294400,
      "gmtoffset": -14400
    },
    "post": {
      "timezone": "EDT",
      "start": 17812

## 3) TEST Social posts from X/Reddit
 * https://docs.x.ai/developers/tools/x-search
 * https://docs.x.ai/developers/tools/citations
 * IMPORTANT! We're using the OpenAI's compatible function call (Responses API) rather than what you can read on the x-search page

In [35]:
import openai

In [36]:
def get_recent_social_posts(ticker: str, max_posts: int = 5, days_back: int = 1):
    """
    Enhanced version with more x_search parameters.
    """
    
    api_key = os.getenv('XAI_API_KEY')
    if not api_key:
        return {
            'ticker': ticker,
            'error': 'XAI_API_KEY not found',
            'posts': []
        }
    
    try:
        from datetime import datetime, timedelta
        
        client = openai.OpenAI(
            api_key=api_key,
            base_url="https://api.x.ai/v1"
        )
        
        # Calculate date range
        to_date = datetime.now()
        from_date = to_date - timedelta(days=days_back)
        
        # Enhanced search query
        search_query = f"""
        Search X/Twitter for recent posts about ${ticker} stock.
        
        Requirements:
        - Find {max_posts} posts from the last {days_back} day(s)
        - Sort by most recent timestamp first
        - Include exact post times (e.g., "2 hours ago", "45 minutes ago")
        - Provide real, working X.com links
        - Include post content, author handles, engagement metrics
        - Analyze sentiment (bullish/bearish/neutral)
        
        Focus on posts about:
        - Stock price movements
        - Company news reactions
        - Earnings discussions
        - Technical analysis
        - Market sentiment
        
        Date range: {from_date.strftime('%Y-%m-%d')} to {to_date.strftime('%Y-%m-%d')}
        """
        
        response = client.responses.create(
            model="grok-4.3",
            input=search_query,
            tools=[{"type": "x_search"}]
        )
        
        # Extract response content
        content = ""
        for item in response.output:
            if item.type == "message":
                for content_block in item.content:
                    if content_block.type == "output_text":
                        content += content_block.text
        
        return {
            'ticker': ticker,
            'response': content,
            'posts': [],
            'search_params': {
                'max_posts': max_posts,
                'days_back': days_back,
                'date_range': f"{from_date.strftime('%Y-%m-%d')} to {to_date.strftime('%Y-%m-%d')}"
            },
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'api_used': 'responses_api_with_x_search'
        }
        
    except Exception as e:
        return {
            'ticker': ticker,
            'error': f'Error: {str(e)}',
            'posts': []
        }

In [37]:
# We'll test the news retrieval and search functionality with a specific ticker and a limit on the number of posts to retrieve.
max_posts = 10
ticker = 'TSLA'

# Test it
print("Testing X.AI ...")
result = get_recent_social_posts(ticker, max_posts)
if 'error' in result:
    print(f"❌ {result['error']}")
else:
    print(f"✅ Got response for {result['ticker']}")

Testing X.AI ...
✅ Got response for TSLA


In [38]:
print(result['response'])

**Here are 10 of the most recent X posts about $TSLA** (from June 13, 2026, sorted by most recent timestamp first). These focus on stock price action, technical analysis, market sentiment, and related news. All links are real/working X.com URLs based on the post IDs returned.

1. **@OGemHODL** (23:58:51 GMT, ~20–30 min ago)  
   Content: "TESLA IS CHARGING UP! MATURING BULL STRUCTURE ON TSLA 🚗⚡ SIGNAL DETAILS: $TSLA /USDT Direction: LONG 🟢 ... Target 1: 419.44 ... Stop Loss: 385.00" (with chart)  
   Engagement: 0 likes, 1 repost, 0 quotes, 0 replies, 44 views  
   Sentiment: **Bullish** (long setup with high profit targets).  
   Link: https://x.com/OGemHODL/status/2065947037057888705

2. **@Robert46989257** (23:58:10 GMT)  
   Content: "lÄnD oF frEE speeecH so they 'asked' twitter to get his adress ? thats pure Gestapo stuff... $TSLA $NVDA $NOW $PLTR" (replying to ICE/FBI post)  
   Engagement: 0 likes, 0 reposts, 0 quotes, 0 replies, 159 views  
   Sentiment: **Neutral** (mentions $

In [39]:
# Search with SORT

def get_recent_social_posts_with_xai(ticker: str, max_posts: int = 5, sort_by: str = "engagement"):
    """
    Get recent Twitter/X posts using xAI's Responses API with x_search tool.
    
    Args:
        ticker: Stock ticker symbol
        max_posts: Number of posts to return
        sort_by: "engagement" (views+likes+replies) or "recent" (timestamp)
    """
    
    api_key = os.getenv('XAI_API_KEY')
    if not api_key:
        return {
            'ticker': ticker,
            'error': 'XAI_API_KEY not found. Set it in your .envrc file.',
            'posts': []
        }
    
    try:
        client = openai.OpenAI(
            api_key=api_key,
            base_url="https://api.x.ai/v1"
        )
        
        # Build search query based on sorting preference
        if sort_by == "engagement":
            sort_instruction = """
            IMPORTANT: Sort by HIGHEST ENGAGEMENT (total impact) first.
            - Calculate engagement score = views + likes + replies + retweets
            - Show the posts with highest engagement numbers first
            - These are the most impactful/viral posts about the stock
            """
        else:
            sort_instruction = "Sort by most recent timestamp first."
        
        search_query = f"""
        Search X/Twitter for posts about ${ticker} stock from the last 24 hours.
        
        {sort_instruction}
        
        For each of the {max_posts} posts, provide:
        - Post content
        - Author username
        - Exact engagement metrics: views, likes, replies, retweets, bookmarks
        - Timestamp
        - Real working X.com link
        - Sentiment analysis
        
        Focus on posts with substantial engagement that show real market impact.
        Skip posts with 0 or minimal engagement unless specifically requested.
        
        Format each post clearly showing the engagement numbers prominently.
        """
        
        response = client.responses.create(
            model="grok-4.3",
            input=search_query,
            tools=[{"type": "x_search"}]
        )
        
        # Extract response content
        content = ""
        for item in response.output:
            if item.type == "message":
                for content_block in item.content:
                    if content_block.type == "output_text":
                        content += content_block.text
        
        return {
            'ticker': ticker,
            'response': content,
            'posts': [],
            'sort_by': sort_by,
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'api_used': 'responses_api_with_x_search'
        }
        
    except openai.AuthenticationError:
        return {
            'ticker': ticker,
            'error': 'Authentication failed. Check your XAI_API_KEY.',
            'posts': []
        }
    
    except openai.RateLimitError:
        return {
            'ticker': ticker,
            'error': 'Rate limit exceeded. Try again later.',
            'posts': []
        }
    
    except Exception as e:
        return {
            'ticker': ticker,
            'error': f'API error: {str(e)}',
            'posts': []
        }

# Function for high-engagement posts specifically
def get_viral_social_posts(ticker: str, max_posts: int = 5, min_engagement: int = 100):
    """
    Get posts with high engagement (viral posts) about a stock.
    """
    
    api_key = os.getenv('XAI_API_KEY')
    if not api_key:
        return {
            'ticker': ticker,
            'error': 'XAI_API_KEY not found',
            'posts': []
        }
    
    try:
        client = openai.OpenAI(
            api_key=api_key,
            base_url="https://api.x.ai/v1"
        )
        
        viral_search_query = f"""
        Search X/Twitter for HIGH-ENGAGEMENT posts about ${ticker} stock from the last 48 hours.
        
        CRITERIA:
        - Only show posts with {min_engagement}+ total engagement (views + likes + replies + retweets)
        - Sort by HIGHEST engagement first (most viral/impactful)
        - Focus on posts that are actually moving opinions or generating discussion
        - Skip spam, bot posts, or posts with minimal engagement
        
        Return the top {max_posts} most viral posts with:
        - Total engagement score prominently displayed
        - Post content 
        - Author (prioritize verified accounts, influencers, analysts)
        - All engagement metrics broken down
        - Working X.com links
        - Why this post is significant/viral
        
        These should be posts that are actually influencing ${ticker} sentiment in the market.
        """
        
        response = client.responses.create(
            model="grok-4.3",
            input=viral_search_query,
            tools=[{"type": "x_search"}]
        )
        
        # Extract content
        content = ""
        for item in response.output:
            if item.type == "message":
                for content_block in item.content:
                    if content_block.type == "output_text":
                        content += content_block.text
        
        return {
            'ticker': ticker,
            'response': content,
            'posts': [],
            'search_type': 'viral_posts',
            'min_engagement': min_engagement,
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
        
    except Exception as e:
        return {
            'ticker': ticker,
            'error': f'Error: {str(e)}',
            'posts': []
        }

In [40]:
print("=== Testing Engagement-Sorted Posts ===")
result1 = get_recent_social_posts_with_xai("TSLA", 5, sort_by="engagement")
if 'error' in result1:
    print(f"❌ Error: {result1['error']}")
else:
    print(f"✅ Success - sorted by {result1['sort_by']}")
    print("Top engagement posts:")

=== Testing Engagement-Sorted Posts ===
✅ Success - sorted by engagement
Top engagement posts:


In [41]:
print(result1['response'])

**Top 5 posts about $TSLA from the last 24 hours (sorted by highest engagement score = views + likes + replies + reposts):**

**1. Highest engagement**  
**Author:** @Realcoinforge  
**Content:** 🇺🇸 TESLA HAS JUST SHOWN A MASSIVE BULL SIGNAL BY BREAKING A TRENDLINE. Every time $TSLA broke a trendline on the higher time-frame, it led to a massive expansion. It has just happened again and if the pattern continues, we will see Tesla at $500 real soon. I believe the next 5 years will be game-changing. Tesla has huge plans for the future.  
**Engagement:** Views: 13,784 | Likes: 116 | Replies: 5 | Reposts: 7 | Bookmarks: 13  
**Timestamp:** Sat, 13 Jun 2026 17:00:16 GMT  
**Link:** https://x.com/Realcoinforge/status/2065841698203422976  
**Sentiment:** Strongly bullish (highlighting a technical breakout and long-term upside to $500+).

**2.**  
**Author:** @peterli34923561  
**Content:** $TSLA --- $TSLA has officially disclosed it holds approximately 19 million shares of SpaceX. Valued at $

In [42]:
print("\n=== Testing Viral Posts (High Engagement Only) ===")
result2 = get_viral_social_posts("TSLA", 5, min_engagement=100)
if 'error' in result2:
    print(f"❌ Error: {result2['error']}")
else:
    print(f"✅ Success - viral posts with 50+ engagement")
    print("Viral posts:")


=== Testing Viral Posts (High Engagement Only) ===
✅ Success - viral posts with 50+ engagement
Viral posts:


In [43]:
print(result1['response'])

**Top 5 posts about $TSLA from the last 24 hours (sorted by highest engagement score = views + likes + replies + reposts):**

**1. Highest engagement**  
**Author:** @Realcoinforge  
**Content:** 🇺🇸 TESLA HAS JUST SHOWN A MASSIVE BULL SIGNAL BY BREAKING A TRENDLINE. Every time $TSLA broke a trendline on the higher time-frame, it led to a massive expansion. It has just happened again and if the pattern continues, we will see Tesla at $500 real soon. I believe the next 5 years will be game-changing. Tesla has huge plans for the future.  
**Engagement:** Views: 13,784 | Likes: 116 | Replies: 5 | Reposts: 7 | Bookmarks: 13  
**Timestamp:** Sat, 13 Jun 2026 17:00:16 GMT  
**Link:** https://x.com/Realcoinforge/status/2065841698203422976  
**Sentiment:** Strongly bullish (highlighting a technical breakout and long-term upside to $500+).

**2.**  
**Author:** @peterli34923561  
**Content:** $TSLA --- $TSLA has officially disclosed it holds approximately 19 million shares of SpaceX. Valued at $

In [44]:
import os
from datetime import datetime
import openai

def get_reddit_stock_discussions(ticker: str, max_posts: int = 5, min_engagement: int = 50):
    """
    Get high-engagement Reddit discussions about a stock using web_search.
    """
    
    api_key = os.getenv('XAI_API_KEY')
    if not api_key:
        return {
            'ticker': ticker,
            'error': 'XAI_API_KEY not found. Set it in your .envrc file.',
            'posts': []
        }
    
    try:
        client = openai.OpenAI(
            api_key=api_key,
            base_url="https://api.x.ai/v1"
        )
        
        # Reddit-focused search query
        reddit_query = f"""
        Search Reddit for high-quality discussions about ${ticker} stock from the last 7 days.
        
        **TARGET SUBREDDITS (search these specifically):**
        - r/investing (serious investment analysis)
        - r/stocks (general stock discussion)
        - r/SecurityAnalysis (fundamental analysis)
        - r/ValueInvesting (value perspective)
        - r/wallstreetbets (retail sentiment & options)
        - r/StockMarket (market discussion)
        - r/financialindependence (long-term investing)
        - Ticker-specific subs if they exist (like r/Tesla for TSLA)
        
        **ENGAGEMENT CRITERIA:**
        - Posts with {min_engagement}+ upvotes OR 30+ comments
        - Comments with 20+ upvotes
        - Awarded posts (Gold, Silver, Helpful, etc.)
        - Active discussions (not just single comments)
        
        **CONTENT TYPES TO PRIORITIZE:**
        1. DD (Due Diligence) posts with detailed analysis
        2. Earnings reaction/discussion threads
        3. News reaction posts with substantial comments
        4. Technical/fundamental analysis posts
        5. Company catalyst discussions
        6. Valuation debates
        
        **OUTPUT FORMAT for each post:**
        
        **[Subreddit Name]** - Post Title
        **Content:** Brief summary of post + key insights from top comments
        **Stats:** X upvotes, Y comments, Z awards
        **Link:** Direct Reddit URL
        **Sentiment:** Bullish/Bearish/Neutral with reasoning
        **Key Points:** 2-3 main takeaways from the discussion
        **Quality Level:** High/Medium (based on depth of analysis)
        
        Return the {max_posts} most valuable Reddit discussions about ${ticker}, sorted by engagement and discussion quality.
        Focus on posts that provide real investment insights, not just price speculation.
        """
        
        response = client.responses.create(
            model="grok-4.3",
            input=reddit_query,
            tools=[{"type": "web_search"}]
        )
        
        # Extract content
        content = ""
        for item in response.output:
            if item.type == "message":
                for content_block in item.content:
                    if content_block.type == "output_text":
                        content += content_block.text
        
        return {
            'ticker': ticker,
            'response': content,
            'posts': [],
            'platform': 'Reddit',
            'min_engagement': min_engagement,
            'search_timeframe': 'last_7_days',
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
        
    except openai.AuthenticationError:
        return {
            'ticker': ticker,
            'error': 'Authentication failed. Check your XAI_API_KEY.',
            'posts': []
        }
    
    except openai.RateLimitError:
        return {
            'ticker': ticker,
            'error': 'Rate limit exceeded. Try again later.',
            'posts': []
        }
    
    except Exception as e:
        return {
            'ticker': ticker,
            'error': f'API error: {str(e)}',
            'posts': []
        }

# Specialized function for DD posts only
def get_reddit_dd_posts(ticker: str, max_posts: int = 3):
    """
    Get only DD (Due Diligence) posts about a stock from Reddit.
    """
    
    api_key = os.getenv('XAI_API_KEY')
    if not api_key:
        return {
            'ticker': ticker,
            'error': 'XAI_API_KEY not found',
            'posts': []
        }
    
    try:
        client = openai.OpenAI(
            api_key=api_key,
            base_url="https://api.x.ai/v1"
        )
        
        dd_query = f"""
        Search Reddit specifically for DD (Due Diligence) posts about ${ticker} stock.
        
        **SEARCH FOCUS:**
        - Look for posts tagged with "DD" or containing detailed financial analysis
        - Search r/investing, r/SecurityAnalysis, r/ValueInvesting, r/stocks
        - Find posts with comprehensive company analysis, not just opinions
        
        **DD POST CRITERIA:**
        - Contains financial metrics, ratios, or analysis
        - Discusses business fundamentals, competition, or growth prospects
        - Has substantial content (not just a few paragraphs)
        - Generated meaningful discussion in comments
        
        **OUTPUT for each DD post:**
        
        **[Subreddit]** - DD Title
        **Author:** Username (mention if they have credibility/history)
        **Analysis Summary:** Key points from the DD (financials, valuation, thesis)
        **Community Response:** Top comments reactions, counterpoints, additional insights
        **Stats:** Upvotes, comments, awards
        **Link:** Reddit URL
        **Investment Thesis:** Bull/bear case summary
        **Quality Score:** High/Medium/Low based on depth and accuracy
        
        Return the {max_posts} highest quality DD posts about ${ticker} from the last 30 days.
        """
        
        response = client.responses.create(
            model="grok-4.3",
            input=dd_query,
            tools=[{"type": "web_search"}]
        )
        
        # Extract content
        content = ""
        for item in response.output:
            if item.type == "message":
                for content_block in item.content:
                    if content_block.type == "output_text":
                        content += content_block.text
        
        return {
            'ticker': ticker,
            'response': content,
            'posts': [],
            'search_type': 'dd_posts_only',
            'platform': 'Reddit',
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
        
    except Exception as e:
        return {
            'ticker': ticker,
            'error': f'Error: {str(e)}',
            'posts': []
        }

In [45]:

# Test the Reddit functions
print("=== Testing Reddit Stock Discussions ===")
result4 = get_reddit_stock_discussions("TSLA", 5, min_engagement=50)

if 'error' in result4:
    print(f"❌ Error: {result4['error']}")
else:
    print(f"✅ Success for {result4['ticker']} on {result4['platform']}")
    print(f"Min engagement: {result4['min_engagement']}")
    print(f"Timeframe: {result4['search_timeframe']}")
    print("\nReddit discussions:")
    print(result4['response'])

=== Testing Reddit Stock Discussions ===
✅ Success for TSLA on Reddit
Min engagement: 50
Timeframe: last_7_days

Reddit discussions:
**Here are the 5 most valuable recent Reddit discussions about $TSLA (from the last 7 days, primarily June 7–14, 2026) that meet the engagement and quality criteria.** Direct standalone TSLA DD, earnings, or analysis threads were limited in the target subreddits during this period. The dominant theme was the SpaceX IPO (around June 12) and frequent comparisons to TSLA’s valuation, with insights on overvaluation, hype vs. fundamentals, PE ratios (~350–400x), and potential merger implications. These provide substantive investment discussion rather than pure price speculation.[[1]](https://www.reddit.com/r/wallstreetbets/comments/1u0bojo/spacex_is_buying_tesla_double_down/)[[2]](https://www.reddit.com/r/investing/comments/1u4lsto/the_math_isnt_mathing_on_the_spacex_ipo/)

**[r/wallstreetbets]** - SpaceX is buying Tesla - Double Down  
**Content:** Detailed o

In [46]:
print("\n=== Testing DD Posts Only ===")
dd_result = get_reddit_dd_posts("TSLA", 2)

if 'error' in dd_result:
    print(f"❌ Error: {dd_result['error']}")
else:
    print(f"✅ Success - {dd_result['search_type']}")
    print("DD posts preview:")
    print(dd_result['response'])
    


=== Testing DD Posts Only ===
✅ Success - dd_posts_only
DD posts preview:
**No high-quality DD posts about $TSLA matching the specified criteria were found in the last 30 days (post ~May 15, 2026) across r/stocks, r/investing, r/ValueInvesting, r/SecurityAnalysis, or related subs.**[[1]](https://www.reddit.com/r/stocks/comments/1u2q6vc/spacex_things_to_consider/)

Extensive web searches using site:reddit.com operators, date filters, keywords like "DD", "due diligence", "financial analysis", financial metrics/ratios, and Tesla/TSLA focus returned mostly tangential mentions, comments in unrelated threads (e.g., SpaceX comparisons, general market discussions), daily/earnings threads, or older posts. No posts featured substantial original financial modeling, ratios (e.g., detailed P/E, FCF, valuation multiples with projections), business fundamentals breakdowns, competition analysis, or growth thesis with meaningful comment engagement that met the depth criteria.[[2]](https://www.reddit.c

# 4) TESTING Edgar and SEC filings
* https://edgartools.readthedocs.io/en/latest/

In [47]:
import os
from edgar import *
set_identity(os.getenv('SEC_IDENTITY_EMAIL'))

In [48]:
company = Company("TSLA")
financials = company.get_financials()
financials.income_statement()

                                                                                                                   
                                         TESLA, INC.   TSLA                                                        
                                         CONSOLIDATED STATEMENT OF INCOME                                          
                                         Dec 31, 2023 to Dec 31, 2025                                              
                                                                                                                   
                                                                     Dec 31, 2025   Dec 31, 2024   Dec 31, 2023    
   ─────────────────────────────────────────────────────────────────────────────────────────────────────────────   
        Revenues                                                                                                   
              Revenues:                                                 

In [51]:
# Get the latest 10-K filing for the company 
filing = company.get_filings(form="10-K").latest()
text_annual = filing.text()  # Clean, readable text

In [52]:
print(text_annual[0:10000])

UNITED STATES

SECURITIES AND EXCHANGE COMMISSION

Washington, D.C. 20549

FORM 10-K/A

(Amendment No. 1)

(Mark One)

  x       ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934  
 ────────────────────────────────────────────────────────────────────────────────────────────────

For the fiscal year ended December 31, 2025

OR

  TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934  
 ────────────────────────────────────────────────────────────────────────────────────────────

For the transition period from

Commission File Number: 001-34756

Tesla, Inc.

(Exact name of registrant as specified in its
charter)

  Texas                                         91-2197729           
  (State or other jurisdiction of               (I.R.S. Employer     
  1 Tesla Road                                  78725                
  (Address of principal executive offices)      (Zip Code)           

( 512 516-8177

(Registrant’s t

In [53]:
filing = company.get_filings(form="10-Q").latest()
text_quarterly = filing.text()  # Clean, readable text

In [54]:
# Initialize OpenAI client - OPENAI_API_KEY env. variable must be set in the .envrc file
client = openai.OpenAI()

In [55]:
# Use a portion of the text to stay within context limits if necessary
# The current 'text' variable holds the latest 10-Q filing from previous cells
input_text = text_annual

response = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {"role": "system", "content": "You are a financial analyst."},
        {"role": "user", "content": f"Analyze the following company filing and identify what is new or significant compared to previous periods. Here is the text: {input_text}"}
    ]
)

print(response.choices[0].message.content)

Here are the main **new or significant items** in this 10-K/A versus prior periods and Tesla’s earlier disclosures:

## 1) This amendment is primarily a Part III “late proxy” filing
- Tesla filed its original 2025 10-K on Jan. 29, 2026, but omitted **Part III** because the proxy statement was not ready.
- This 10-K/A adds:
  - **Directors / executive officers / governance**
  - **Executive compensation**
  - **Ownership**
  - **Related-party transactions**
  - **Auditor fees**
- Tesla also filed **new CEO/CFO certifications** with the amendment.

## 2) Major new board and leadership changes
### New director: **Jack Hartung**
- Added to the board in **June 2025**.
- Former senior advisor / president at **Chipotle**.
- Notable because this is a relatively new director with strong finance, strategy, and operations credentials.

### Board / committee structure update
- The board now includes Jack Hartung on the **Audit Committee**.
- Tesla’s disclosure confirms the board still has four sta

In [67]:
def get_usage_info_chatcompletion(response):
    """
    Extract usage information and estimate API cost from a ChatCompletion response.
    """

    # GPT-5.4-mini pricing
    INPUT_PRICE = 0.75 / 1_000_000
    OUTPUT_PRICE = 4.50 / 1_000_000

    if not hasattr(response, "usage") or response.usage is None:
        return {"error": "Usage information not available."}

    usage = response.usage

    input_tokens = usage.prompt_tokens
    output_tokens = usage.completion_tokens

    return {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": usage.total_tokens,
        "cost_usd": (
            input_tokens * INPUT_PRICE +
            output_tokens * OUTPUT_PRICE
        ),
        "prompt_tokens_details": (
            usage.prompt_tokens_details.model_dump()
            if hasattr(usage.prompt_tokens_details, "model_dump")
            else None
        ),
        "completion_tokens_details": (
            usage.completion_tokens_details.model_dump()
            if hasattr(usage.completion_tokens_details, "model_dump")
            else None
        ),
    }

In [68]:
# $0.03 cost of analysis
get_usage_info_chatcompletion(response)

{'input_tokens': 30673,
 'output_tokens': 1883,
 'total_tokens': 32556,
 'cost_usd': 0.03147825,
 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
 'completion_tokens_details': {'accepted_prediction_tokens': 0,
  'audio_tokens': 0,
  'reasoning_tokens': 0,
  'rejected_prediction_tokens': 0}}

In [72]:
# Use a portion of the text to stay within context limits if necessary
# The current 'text' variable holds the latest 10-Q filing from previous cells
input_text_quarterly = text_quarterly
input_text_annual = text_annual


response_full = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {"role": "system", "content": "You are a financial analyst."},
        {"role": "user", "content": f"Analyze the following company filing and identify what is new or significant compared to previous periods. \
          Here is the latest quarterly text: {input_text_quarterly}. Here is the latest annual report released earlier: {input_text_annual}.\
          Turn this into a **“new vs. prior period” table** with columns for **change / financial impact / why it matters. Provide a concise summary of the key points and any potential implications for investors."}
    ]
)

print(response_full.choices[0].message.content)

Below is a concise **new vs. prior period** table highlighting what changed in Tesla’s **Q1 2026 10-Q** versus **Q1 2025** and, where relevant, versus **year-end 2025**. I’ve focused on items that are **new, materially different, or strategically significant**.

## Tesla Q1 2026: New vs. Prior Period

| Change | Financial impact | Why it matters |
|---|---:|---|
| **Revenue reaccelerated to $22.4B, up 16% YoY** | +$3.05B revenue | Top-line growth improved, driven mainly by higher automotive sales and services. |
| **Automotive sales rose 20% YoY** | +$2.55B | Suggests higher deliveries and better average selling price; management said deliveries rose ~10% and pricing improved from mix and FX. |
| **Regulatory credits fell 36% YoY** | -$215M | Tesla is becoming less reliant on credit sales; good if core auto profitability is improving, but it removes a margin-supporting revenue source. |
| **Services and other revenue jumped 42% YoY** | +$1.11B | Strong growth in used vehicle sales, mai

In [73]:
# readable output in Jupyter Notebook
from IPython.display import Markdown, display
display(Markdown(response_full.choices[0].message.content))

Below is a concise **new vs. prior period** table highlighting what changed in Tesla’s **Q1 2026 10-Q** versus **Q1 2025** and, where relevant, versus **year-end 2025**. I’ve focused on items that are **new, materially different, or strategically significant**.

## Tesla Q1 2026: New vs. Prior Period

| Change | Financial impact | Why it matters |
|---|---:|---|
| **Revenue reaccelerated to $22.4B, up 16% YoY** | +$3.05B revenue | Top-line growth improved, driven mainly by higher automotive sales and services. |
| **Automotive sales rose 20% YoY** | +$2.55B | Suggests higher deliveries and better average selling price; management said deliveries rose ~10% and pricing improved from mix and FX. |
| **Regulatory credits fell 36% YoY** | -$215M | Tesla is becoming less reliant on credit sales; good if core auto profitability is improving, but it removes a margin-supporting revenue source. |
| **Services and other revenue jumped 42% YoY** | +$1.11B | Strong growth in used vehicle sales, maintenance, Supercharging, and insurance indicates the ecosystem business is expanding. |
| **Energy generation & storage revenue declined 12% YoY** | -$322M | A softer energy quarter, likely from lower Megapack/Powerwall deployments; important because this is a key growth segment. |
| **Gross margin improved sharply to 21.1% from 16.3%** | +4.8 pts total gross margin | Indicates better pricing, mix, and/or cost control; a major positive for profitability. |
| **Automotive gross margin improved to 21.1% from 16.2%** | +4.9 pts | Shows core auto profitability improved materially even with lower regulatory credits. |
| **Energy gross margin rose to 39.5% from 28.8%** | +10.7 pts | Suggests much better unit economics in energy, despite lower revenue. |
| **Operating income more than doubled to $941M** | +$542M | Indicates operating leverage is improving. |
| **R&D increased 38% YoY** | +$537M | Tesla is spending more on AI, autonomy, robotics, and new programs; this is strategically important but raises near-term expense load. |
| **SG&A increased 47% YoY** | +$582M | Driven by stock comp, labor, and legal costs. This is a margin headwind and may persist if legal/compensation costs remain elevated. |
| **Other expense, net worsened sharply** | -$416M vs. prior year | Mainly foreign exchange losses and digital asset mark-to-market losses; adds earnings volatility. |
| **Effective tax rate increased to 34% from 29%** | Higher tax expense (+$88M tax provision) | Partly due to jurisdiction mix and non-deductibility of stock comp tied to CEO awards; reduces bottom-line conversion. |
| **Operating cash flow improved to $3.94B** | +$1.78B YoY | Strong cash generation supports capex and AI investment plans. |
| **Capex rose to $2.49B** | +$1.00B YoY | Tesla is spending more on AI infrastructure, manufacturing, and factory expansion; signals aggressive investment mode. |
| **Tesla bought $2.0B of SpaceX equity** | New investing cash outflow of $2.0B | This is a major new related-party investment and a notable use of cash; investors may view it as strategic but non-core. |
| **Debt structure changed significantly** | Total debt/finance leases fell to $9.23B from $8.39B? (note: balance sheet current+LT debt/leases rose to $9.23B from $8.38B) | Tesla increased financing activity in Q1 2026, including new debt proceeds and a new warehouse facility. |
| **New Warehouse Agreement entered in Q1 2026** | Potential borrowing capacity up to $1.5B | Expands financing flexibility; could support vehicle/asset-backed funding needs. |
| **2018 CEO Performance Award reinstated/implemented** | No new P&L expense for 2018 award, but major governance impact | A major corporate governance event: Tesla and Elon Musk entered the “Implementation Agreement,” changing how the award can be exercised. |
| **2025 CEO Interim Award forfeited in April 2026** | Removes a previously disclosed $26.1B grant-date accounting value from future compensation discussions | This is a major compensation and governance update. It eliminates a potentially massive outstanding CEO award. |
| **2025 CEO Performance Award remains outstanding; no tranches earned yet** | No current cash impact, but large potential future dilution/comp | Still one of the largest contingent compensation awards ever disclosed; important for dilution and governance scrutiny. |
| **New acquisition agreement for an AI hardware company** | Up to $2.0B in Tesla stock/equity awards (mostly service/performance contingent) | Signals continued push into AI hardware/compute; could expand capabilities but adds integration and execution risk. |
| **Tariff ruling may allow refunds** | Potential future benefit, not yet recognized | If recovered, could lower costs or increase income, but timing/amount is uncertain. |
| **Legal proceedings progressed** | Several cases resolved or advanced; no single quantified new reserve disclosed | Notable developments: Delaware derivative settlement affirmed and fee award reduced; some suits dismissed or moving toward trial; continued litigation risk remains. |

---

## Key new / significant items by category

### 1) Business performance
- **Revenue, gross profit, operating income, and operating cash flow all improved.**
- The main positives were:
  - **higher automotive sales**
  - **much better gross margins**
  - **stronger services revenue**
- The main drag was:
  - **weaker energy revenue**
  - **higher operating expenses**
  - **foreign exchange and digital asset losses**

### 2) Strategic shifts
- Tesla is clearly emphasizing **AI, autonomy, robotics, and compute infrastructure**.
- Management explicitly said it is focused on:
  - **FSD (Supervised)**
  - **Robotaxi/Cybercab**
  - **Optimus**
  - **semiconductor fabrication**
  - **AI data centers / training clusters**
- Capex guidance of **more than $25B in 2026** is a major indicator of this shift.

### 3) Related-party investment
- Tesla invested **$2.0B in SpaceX** in Q1 2026.
- This is significant because it is:
  - a large cash deployment
  - related-party in nature
  - non-core to the automotive business
- Investors may view it as strategically interesting but also as a use of capital outside Tesla’s main operations.

### 4) CEO compensation / governance
This is the biggest governance story in the filing:
- The **2018 CEO Performance Award was reinstated** by the Delaware Supreme Court and then operationalized through a new **Implementation Agreement**.
- The **2025 CEO Interim Award was forfeited** in April 2026.
- Tesla says the Implementation Agreement creates **no additional economic benefit**, but it does impose a **service-based vesting condition** and a **holding period** on shares issued upon exercise.
- This reduces immediate windfall concerns but keeps CEO comp and dilution in focus.

### 5) Financing / balance sheet
- Tesla’s liquidity remains strong:
  - **$44.7B cash + short-term investments**
  - **$5.0B unused committed credit**
- But there are notable movements:
  - new debt issuance
  - repayment activity
  - new warehouse facility
  - higher capex and related investment spending

---

## Investor implications

### Positive takeaways
- **Core profitability improved materially**
- **Operating cash flow strengthened**
- **Services revenue is scaling**
- **Energy gross margin improved sharply**
- **Strong liquidity supports heavy AI and manufacturing investment**

### Watch items / risks
- **Energy revenue weakened**
- **R&D and SG&A are rising quickly**
- **Other expense remains volatile**
- **CEO compensation/governance remains a major overhang**
- **Large AI and robotics spending could pressure near-term free cash flow**
- **Related-party investments and litigation may keep investor scrutiny high**

## Bottom line
Tesla’s Q1 2026 filing is notable for:
1. **better operating performance and margins**,  
2. **heavier investment in AI/autonomy/robotics**, and  
3. **major CEO compensation/governance developments** that materially change the stock-comp picture.

If you want, I can also convert this into a **bullet-style earnings note**, a **risk/opportunity matrix**, or a **“what changed in the 10-Q vs 10-K” redline summary**.

In [74]:
# $0.05 cost of analysis
get_usage_info_chatcompletion(response_full)

{'input_tokens': 56633,
 'output_tokens': 1896,
 'total_tokens': 58529,
 'cost_usd': 0.05100675,
 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
 'completion_tokens_details': {'accepted_prediction_tokens': 0,
  'audio_tokens': 0,
  'reasoning_tokens': 0,
  'rejected_prediction_tokens': 0}}